# SEC598 — How LLMs "Read" Text: Tokens, Embeddings & Vectors

*Companion notebook to the lecture slides on tokenization and embeddings.*

An LLM never sees your words — it sees **numbers**. This notebook walks that
conversion end to end, with real models (so the token IDs are the actual ones,
not illustrative):

1. **Tokens & IDs** — turn text into tokens, compare BPE vs. WordPiece.
2. **Embeddings** — turn tokens into vectors.
3. **Visualize** — similarity, analogies, and a 2-D map where related meanings cluster.

**How to run:** `Runtime → Run all`, or run each cell with `Shift+Enter`, top to bottom.
The first run installs packages and downloads a small model — give it a minute.

In [ ]:
# --- Setup: install the pieces we need (Colab already has torch, sklearn, matplotlib) ---
!pip -q install tiktoken transformers gensim
print("Setup complete. (If a cell later errors on gensim, run:")
print("  !pip install --upgrade gensim scipy   then  Runtime -> Restart runtime )")

---
## Part 1 — Tokenization: text → tokens → IDs

Tokenization is the **first step** of any NLP pipeline. Text is broken into
*tokens* (words, subwords, or characters), and each token is mapped to an
integer **ID** from a fixed vocabulary. The model computes on those IDs.

Below we use `cl100k_base` — the Byte-Pair-Encoding (BPE) tokenizer behind
GPT-3.5 / GPT-4.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")   # the tokenizer behind GPT-3.5 / GPT-4

text = "LLMs read tokens, not words."
ids  = enc.encode(text)

print(f"Text:        {text!r}")
print(f"Token count: {len(ids)}")
print(f"Token IDs:   {ids}\n")

print(f"{'TOKEN':<14}{'ID':>8}")
print("-" * 22)
for tid in ids:
    piece = enc.decode([tid])
    print(f"{piece!r:<14}{tid:>8}")

### Watch words break into *subwords*

Common words are a single token; rare, long, or novel words get split into
smaller known pieces. This is why subword tokenization keeps the vocabulary
manageable while still covering code, URLs, and words it has never seen.

In [ ]:
def show_tokens(text, encoding="cl100k_base"):
    enc = tiktoken.get_encoding(encoding)
    ids = enc.encode(text)
    pieces = [enc.decode([t]) for t in ids]
    print(f"{text!r}  ->  {len(ids)} tokens")
    print("  " + " | ".join(repr(p) for p in pieces))
    print("  ids: " + str(ids) + "\n")

for s in ["tokenization",
          "antidisestablishmentarianism",
          "def exploit(payload): return payload",
          "https://globexautomation.ai/login",
          "cybersecurity"]:
    show_tokens(s)

# >>> Your turn: paste a log line or command below and run it.
show_tokens("Failed password for root from 10.0.0.5 port 22 ssh2")

### See the tokens: a playground-style view

Same idea as the OpenAI tokenizer playground — each colored box is one token.
Leading spaces are part of the token (that's why `" read"` includes the space),
and you can watch long words get chopped into subword boxes.

In [ ]:
import matplotlib.pyplot as plt

def visualize_tokens(text, encoding="cl100k_base", max_cols=64):
    enc    = tiktoken.get_encoding(encoding)
    ids    = enc.encode(text)
    pieces = [enc.decode([t]) for t in ids]

    palette = ["#ffadad", "#ffd6a5", "#fdffb6", "#caffbf",
               "#9bf6ff", "#a0c4ff", "#bdb2ff", "#ffc6ff"]

    # Lay tokens out left-to-right in a monospace grid, wrapping at max_cols.
    placed, x, y = [], 0, 0
    for i, p in enumerate(pieces):
        disp = p.replace("\n", "\\n").replace("\t", "\\t")
        w = max(len(disp), 1)
        if x + w > max_cols and x > 0:
            x, y = 0, y - 1
        placed.append((x, y, w, disp, palette[i % len(palette)]))
        x += w
    rows = -y + 1

    fig, ax = plt.subplots(figsize=(14, 0.6 * rows + 0.8))
    for x, y, w, disp, color in placed:
        ax.add_patch(plt.Rectangle((x, y), w, 0.9, facecolor=color,
                                   edgecolor="white", linewidth=1.5))
        ax.text(x + w / 2, y + 0.45, disp, ha="center", va="center",
                family="monospace", fontsize=12)
    ax.set_xlim(0, max_cols)
    ax.set_ylim(y - 0.2, 1.1)
    ax.axis("off")
    ax.set_title(f"{text!r}\n{len(ids)} tokens  ({encoding})",
                 fontsize=12, loc="left")
    plt.tight_layout(); plt.show()

visualize_tokens("LLMs read tokens, not words.")
visualize_tokens("Tokenizing cybersecurity vulnerabilities is surprisingly tricky!")

# >>> Your turn: drop in your own text.
visualize_tokens("GET /admin/../../etc/passwd HTTP/1.1  <- path traversal")

### BPE vs. WordPiece

Different models tokenize the *same* sentence differently. Compare OpenAI-style
**BPE** (GPT-2) with Google-style **WordPiece** (BERT). Notice WordPiece marks
word continuations with `##`.

In [ ]:
from transformers import AutoTokenizer

gpt2 = AutoTokenizer.from_pretrained("gpt2")               # BPE        (OpenAI style)
bert = AutoTokenizer.from_pretrained("bert-base-uncased")  # WordPiece  (Google style)

sample = "Tokenizing cybersecurity vulnerabilities"
print("Sample:", sample, "\n")

print(f"GPT-2  (BPE)       vocab = {gpt2.vocab_size}")
print("  tokens:", gpt2.tokenize(sample), "\n")

print(f"BERT   (WordPiece) vocab = {bert.vocab_size}")
print("  tokens:", bert.tokenize(sample))
print("\nWordPiece marks continuations with '##' (e.g. '##izing').")

---
## Part 2 — Embeddings: IDs → vectors

A token ID is just a lookup key. The model turns each ID into an **embedding**:
a list of numbers (a *vector*) that positions the token in a high-dimensional
"meaning space." Tokens with similar meaning get similar vectors.

To make this tangible we'll load small, pretrained **GloVe** word vectors
(50 numbers per word). These are *static* word embeddings — one vector per word
— which are perfect for building intuition. (Part 3 shows the *contextual*
embeddings real LLMs use.)

In [ ]:
import gensim.downloader as api

print("Downloading word vectors (first run only, ~66 MB)...")
wv = api.load("glove-wiki-gigaword-50")
print(f"Loaded {len(wv.key_to_index):,} words, each a {wv.vector_size}-dimension vector.")

def present(words):
    '''Keep only words this vocabulary actually knows (avoids KeyError).'''
    ok = [w for w in words if w in wv.key_to_index]
    missing = [w for w in words if w not in wv.key_to_index]
    if missing:
        print("  (skipped - not in this vocab:", missing, ")")
    return ok

In [ ]:
word = "firewall"
vec = wv[word]
print(f"The word {word!r} is literally this vector:\n")
print(vec)
print(f"\nShape: {vec.shape}  ->  {vec.shape[0]} numbers standing in for one word.")

### Similarity = closeness in meaning space

**Cosine similarity** measures how close two vectors point: `1.0` = same
direction (very similar meaning), `0` = unrelated.

In [ ]:
pairs = [("malware", "virus"),
         ("firewall", "security"),
         ("phishing", "email"),
         ("cat", "dog"),
         ("firewall", "banana")]

print("Cosine similarity between word vectors:")
for a, b in pairs:
    if a in wv.key_to_index and b in wv.key_to_index:
        print(f"  {a:>10} <-> {b:<10}  {wv.similarity(a, b):.3f}")

print("\nNearest neighbors of 'malware':")
for w, score in wv.most_similar("malware", topn=8):
    print(f"  {w:<16}{score:.3f}")

### Meaning as arithmetic

Because meaning is encoded as direction, you can do algebra on words. The classic
result: `king - man + woman` lands near **queen**.

In [ ]:
print("king - man + woman  ~=")
for w, score in wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3):
    print(f"  {w:<10}{score:.3f}")

print("\nparis - france + italy  ~=")
for w, score in wv.most_similar(positive=["paris", "italy"], negative=["france"], topn=3):
    print(f"  {w:<10}{score:.3f}")

### See it: project the vectors to 2-D

These vectors have 50 dimensions — we can't picture that. **PCA** squashes them
to 2-D so we can plot them. Watch related words cluster together, with no labels
telling the algorithm which group they belong to.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

groups = {
    "security":  ["malware", "firewall", "phishing", "exploit", "vulnerability", "encryption"],
    "animals":   ["dog", "cat", "horse", "elephant", "tiger", "rabbit"],
    "countries": ["france", "germany", "japan", "brazil", "egypt", "canada"],
    "food":      ["pizza", "banana", "coffee", "bread", "cheese", "apple"],
}

words, labels = [], []
for g, ws in groups.items():
    for w in present(ws):
        words.append(w); labels.append(g)

X  = np.array([wv[w] for w in words])        # 50-dim vectors
xy = PCA(n_components=2).fit_transform(X)     # squash to 2-D

plt.figure(figsize=(11, 8))
colors = {"security": "#d64545", "animals": "#2f9e44",
          "countries": "#1971c2", "food": "#e8a93c"}
for g in groups:
    idx = [i for i, l in enumerate(labels) if l == g]
    plt.scatter(xy[idx, 0], xy[idx, 1], c=colors[g], label=g, s=90)
for i, w in enumerate(words):
    plt.annotate(w, (xy[i, 0] + 0.05, xy[i, 1] + 0.05), fontsize=10)

plt.title("Word embeddings projected to 2-D — related meanings cluster together")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## Part 3 — Contextual embeddings (what modern LLMs actually use)

GloVe gives each word **one** fixed vector. Transformer LLMs instead produce a
**contextual** vector — the same word gets a *different* vector depending on the
sentence around it. Here we pull BERT's vector for the word *"bank"* in three
sentences and compare them.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

tok   = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased").eval()

def word_vector(sentence, target):
    enc = tok(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = model(**enc).last_hidden_state[0]        # [tokens, 768]
    toks = tok.convert_ids_to_tokens(enc["input_ids"][0])
    return hidden[toks.index(target)]

def cos(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

v_money = word_vector("I deposited cash at the bank.", "bank")
v_river = word_vector("We sat on the river bank.",      "bank")
v_loan  = word_vector("The bank approved my loan.",     "bank")

print("Same WORD 'bank', different CONTEXT -> different vector:")
print(f"  financial vs river : {cos(v_money, v_river):.3f}")
print(f"  financial vs loan  : {cos(v_money, v_loan):.3f}   (both financial -> higher)")

---
## Part 4 — Inside the Transformer: self-attention

So the words are vectors (Part 2) — then what? The Transformer's core move is
**self-attention**: for every token, it looks at *every other token* and weighs
how much each one matters. That's how it captures relationships across a whole
sentence, in parallel.

We can *see* it. BERT exposes its attention weights, so below we plot a
**token × token heatmap**. Read one **row** as: "when the model processes this
token, where does it look?" Brighter = more attention.

> Attention is *suggestive*, not a ground-truth parser — a real model blends
> many heads across many layers. We average the heads of one layer here.

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

attn_tok   = AutoTokenizer.from_pretrained("bert-base-uncased")
attn_model = AutoModel.from_pretrained("bert-base-uncased",
                                       output_attentions=True).eval()

def plot_attention(sentence, layer=6, head=None):
    enc = attn_tok(sentence, return_tensors="pt")
    with torch.no_grad():
        attn = attn_model(**enc).attentions[layer][0]   # [heads, seq, seq]
    A    = (attn[head] if head is not None else attn.mean(0)).numpy()
    toks = attn_tok.convert_ids_to_tokens(enc["input_ids"][0])

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(A, cmap="viridis")
    ax.set_xticks(range(len(toks))); ax.set_xticklabels(toks, rotation=90, fontsize=10)
    ax.set_yticks(range(len(toks))); ax.set_yticklabels(toks, fontsize=10)
    ax.set_xlabel("attended TO  (where it looks)")
    ax.set_ylabel("attending FROM  (this token)")
    htxt = f"head {head}" if head is not None else "mean of all heads"
    ax.set_title(f"Self-attention - layer {layer}, {htxt}\n{sentence!r}",
                 fontsize=10, loc="left")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

# What does "it" look at? (coreference)
plot_attention("The server dropped the connection because it was overloaded.")

# >>> Your turn: change the sentence, the layer (0-11), or pick a single head.
plot_attention("The analyst flagged the alert because it looked malicious.", layer=8)

---
## Part 5 — From vectors to the next word: softmax, temperature & generation

This is the **output side** of the pipeline — the part the 3Blue1Brown video
covers that Parts 1–4 didn't. After all the attention layers, the final vector
becomes a list of raw scores called **logits**, one per vocabulary token:

> final vector  →  **logits**  →  **softmax** → probabilities  →  **sample**  →  repeat

**Softmax** turns arbitrary scores into a probability distribution.
**Temperature** reshapes that distribution (creative vs. predictable).
Then we **sample** a token, append it, and run it all again — that loop *is* text
generation. We use real GPT-2 here, so the logits are the model's actual output.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

gen_tok   = AutoTokenizer.from_pretrained("gpt2")
gen_model = AutoModelForCausalLM.from_pretrained("gpt2").eval()   # GPT-2: a next-token predictor

def next_token_logits(prompt):
    ids = gen_tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        return gen_model(ids).logits[0, -1]        # scores for the NEXT token

prompt = "The firewall blocked the"
logits = next_token_logits(prompt)
probs  = F.softmax(logits, dim=-1)

topk   = torch.topk(probs, 10)
labels = [repr(gen_tok.decode([i])) for i in topk.indices]

plt.figure(figsize=(10, 4))
plt.bar(range(10), topk.values.numpy(), color="#1971c2")
plt.xticks(range(10), labels, rotation=45, ha="right")
plt.ylabel("probability")
plt.title(f"Top-10 next tokens after {prompt!r}  (softmax of the logits)")
plt.tight_layout(); plt.show()

### Temperature reshapes the distribution

Same logits, divided by a **temperature** `T` before softmax. Low `T` sharpens
toward the single most likely token (predictable); high `T` flattens it (more
random). `T → 0` is "always pick the top token"; `T` large approaches uniform.

In [ ]:
def softmax_T(logits, T):
    return F.softmax(logits / T, dim=-1)

# Track the 6 tokens that are most likely at T=1.0, across several temperatures.
base_top = torch.topk(F.softmax(logits, dim=-1), 6).indices
toks = [repr(gen_tok.decode([i])) for i in base_top]
temps = [0.3, 0.7, 1.0, 1.5]

plt.figure(figsize=(11, 4.5))
width = 0.2
for j, T in enumerate(temps):
    p = softmax_T(logits, T)[base_top].numpy()
    plt.bar([x + j * width for x in range(6)], p, width=width, label=f"T={T}")
plt.xticks([x + 1.5 * width for x in range(6)], toks, rotation=45, ha="right")
plt.ylabel("probability"); plt.legend(title="temperature")
plt.title("Temperature reshapes the SAME logits: low T = peaky, high T = flat")
plt.tight_layout(); plt.show()

### The generation loop

Predict → sample → append → repeat. That loop, running on top of everything in
Parts 1–4, is all a language model does to write text.

In [ ]:
def generate(prompt, n_tokens=30, T=0.8, seed=0):
    torch.manual_seed(seed)
    ids = gen_tok(prompt, return_tensors="pt").input_ids
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = gen_model(ids).logits[0, -1]
        nxt = torch.multinomial(F.softmax(logits / T, dim=-1), 1)  # sample one token
        ids = torch.cat([ids, nxt.view(1, 1)], dim=1)              # append it
    return gen_tok.decode(ids[0])

seed_prompt = "In cybersecurity, the most common mistake is"
print("--- T = 0.3 (predictable) ---")
print(generate(seed_prompt, T=0.3), "\n")
print("--- T = 1.2 (creative) ---")
print(generate(seed_prompt, T=1.2))

---
## Recap & exercises

**The pipeline you just ran:**

| Stage | What happened | Cell |
|---|---|---|
| Tokens & IDs | text → subword tokens → integer IDs | Part 1 |
| Embeddings | each ID → a vector in meaning space | Part 2 |
| Meaning | similarity, analogies, 2-D clusters | Part 2 |
| Context | same word → different vector per sentence | Part 3 |
| Attention | each token weighs every other token | Part 4 |
| Next token | final vector → logits → softmax → probabilities → sample | Part 5 |

**Try it yourself**
1. Tokenize one of your own security log lines in Part 1 — how many tokens is an IP address? A hash?
2. In Part 2, run `wv.most_similar("ransomware", topn=10)` — does this 2014-era vocabulary even know the word?
3. Add a `"networking"` group (`router, switch, packet, latency, ...`) to the scatter and re-plot.
4. Change the three *"bank"* sentences in Part 3 to another ambiguous word (e.g. *"patch"*, *"shell"*, *"root"*) — very relevant to security!
5. In Part 4, write a sentence with an ambiguous *"it"* and check what it attends to. Sweep the layer (0–11) and try a single `head=` — different heads learn different jobs.
6. In Part 5, change the prompt to a security scenario and read the top-10 next tokens — is the model confident (one tall bar) or unsure (many short bars)?
7. In Part 5, generate from the same prompt at `T=0.2` vs `T=1.3` — how repetitive vs. wild does the text get?

**Caveat:** GloVe is static and tiny; production LLMs use contextual embeddings
(Part 3) with billions of parameters. The *ideas* are identical — the scale is not.